# prettyprint: Stable Diffusion from scratch on a Colab T4

Pipeline: **VAE -> tokenizer + cached latents -> text-conditioned UNet (CFG, EMA, Min-SNR) -> sampling / img2img / inpainting / LoRA**.

Select **Runtime > Change runtime type > T4 GPU**, then run the cells top to bottom. Everything is written to Google Drive,
so if the session disconnects just re-run the same cells: training resumes from `latest.pt`.

In [ ]:
import os, torch
assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/prettyprint'   # checkpoints, caches and samples live here
os.makedirs(WORK, exist_ok=True)

In [ ]:
!git clone -b rewrite https://github.com/0xC0ba1t/prettyprint.git /content/prettyprint 2>/dev/null || git -C /content/prettyprint pull
%cd /content/prettyprint
!pip install -q -r requirements.txt

## 1. Config
Paths are redirected to Drive. Edit `configs/t4_128.yaml` (or override below) to change model size, steps or dataset.

In [ ]:
import sys; sys.path.insert(0, '/content/prettyprint')
from sd.config import load_config, save_config
cfg = load_config('configs/t4_128.yaml')
cfg.vae_train.out_dir = f'{WORK}/vae'
cfg.train.out_dir = f'{WORK}/ldm'
cfg.lora.out_dir = f'{WORK}/lora'
cfg.data.cache_path = f'{WORK}/cache/latents.pt'
cfg.data.tokenizer_path = f'{WORK}/cache/tokenizer.json'
save_config(cfg, f'{WORK}/config.yaml')
CFG = f'{WORK}/config.yaml'

## 2. Train the VAE (~1-1.5 h for 8k steps on a T4)

In [ ]:
!python scripts/train_vae.py --config $CFG

In [ ]:
from IPython.display import Image, display
import glob
display(Image(sorted(glob.glob(f'{WORK}/vae/recon_*.png'))[-1]))   # top row: input, bottom row: reconstruction

## 3. Train the tokenizer and cache latents

In [ ]:
!python scripts/prepare_data.py --config $CFG --vae $WORK/vae/vae.pt

## 4. Train the diffusion model (resumable)

In [ ]:
!python scripts/train_ldm.py --config $CFG --vae $WORK/vae/vae.pt

In [ ]:
import glob
display(Image(sorted(glob.glob(f'{WORK}/ldm/samples/*.png'))[-1]))

## 5. Text-to-image

In [ ]:
from sd.pipeline import LatentDiffusionPipeline
pipe = LatentDiffusionPipeline.from_checkpoints(f'{WORK}/ldm/latest.pt', f'{WORK}/vae/vae.pt', cfg.data.tokenizer_path)
from sd.utils import make_grid
images = pipe('a blue water monster with big eyes', num_images=4, num_inference_steps=25, guidance_scale=5.0, sampler='dpmpp', seed=1)
make_grid(images)

## 6. img2img and inpainting

In [ ]:
from PIL import Image as PILImage
init = images[0]
edited = pipe.img2img('a red fire monster', init, strength=0.6, num_images=4, seed=2)
make_grid([init.resize(edited[0].size)] + edited)

In [ ]:
mask = PILImage.new('L', init.size, 0)
mask.paste(255, (init.size[0] // 2, 0, init.size[0], init.size[1]))   # white = repaint the right half
inpainted = pipe.inpaint('a yellow electric monster', init, mask, num_images=4, seed=3)
make_grid([init] + inpainted)

## 7. LoRA fine-tuning
Put a few dozen images in a Drive folder (optionally `name.txt` captions next to each file), then train adapters only (a few MB).

In [ ]:
STYLE_DIR = f'{WORK}/my_images'   # <- your images
!python scripts/train_lora.py --ldm $WORK/ldm/latest.pt --vae $WORK/vae/vae.pt --tokenizer $WORK/cache/tokenizer.json --folder $STYLE_DIR

In [ ]:
from sd.lora import load_lora
load_lora(pipe.unet, f'{WORK}/lora/lora.pt', scale=1.0, merge=True)
make_grid(pipe('a small round creature', num_images=4, seed=5))